# Exploration de la méthode de calcul de points par conv2D

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import torch as t
from pytorch_lightning.trainer import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint, TQDMProgressBar, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger, CSVLogger
import optuna

from data_handling.database_service import DatabaseService
from data_handling.conv_method_data_prep import ConvMethodDataPreparator
from points_methods.conv2D_model.net import CompetitionAutoencoderNet
from points_methods.conv2D_model.data_module import CompetitionDataModule
from points_methods.conv2D_model.batch_handling import random_crop_batch_to_smallest

## Préparation des données

Sauvegarde des données préparées sous forme de fichiers HDF5 pour ne pas surcharger la RAM lors de l'entraînement

In [ ]:
files_saving_path = Path.cwd() / "hdf5_data" / "1y_tables"
db_service = DatabaseService()
data_preparator = ConvMethodDataPreparator(
    db_service, competition_table_period=timedelta(days=365)
)
# ONLY ONCE
# data_preparator.save_competition_data_over_period(
#     starting_date=datetime(2001, 1, 1),
#     ending_date=datetime(2025, 3, 1),
#     saving_path=files_saving_path,
#     seed=0,
# )

## Optimisation des hyperparamètres

In [ ]:
def objective(trial):
    nb_hidden_layers = trial.suggest_int("nb_hidden_layers", 1, 3)
    nb_hidden_channels = trial.suggest_categorical(
        "nb_hidden_channels", [8, 16, 32]
    )
    z_dim = trial.suggest_int("z_dim", 2, 8)
    huber_loss_delta = trial.suggest_float("huber_loss_delta", np.log(1.03), np.log(1.2), log=True)

    max_epochs = 15
    batch_size = 4
    competition_table_period = timedelta(days=365)
    train_test_lim = datetime(2023, 1, 1)
    hidden_part = 0.15
    num_workers = 4

    net_hyperparams = dict(
        in_channels_enc=6,
        hidden_channels_enc=nb_hidden_channels,
        nb_hidden_layers_enc=nb_hidden_layers,
        zdim_line=z_dim,
        zdim_col=z_dim,
        hidden_channels_dec=nb_hidden_channels,
        nb_hidden_layers_dec=nb_hidden_layers,
        huber_loss_delta=huber_loss_delta,
        warmup_epochs=3,
        total_epochs=15,
    )

    early_stopping = EarlyStopping(monitor="Median absolute prediction error (s)", patience=5)

    trainer = Trainer(
        callbacks=[early_stopping],
        max_epochs=max_epochs,
        logger=CSVLogger("optuna_logs"),
    )

    data_module = CompetitionDataModule(
        files_saving_path,
        batch_size=batch_size,
        competition_table_period=competition_table_period,
        train_test_lim=train_test_lim,
        hidden_part=hidden_part,
        num_workers=num_workers,
    )

    model = CompetitionAutoencoderNet(**net_hyperparams)

    trainer.fit(model, data_module)
    return trainer.callback_metrics["Median absolute prediction error (s)"].item()


study = optuna.create_study(
    study_name="recherche hyperparametres conv2D",
    storage="sqlite:///conv2D.db",
    load_if_exists=True,
    direction="minimize",
)
#study.optimize(objective, n_trials=10)

In [ ]:
study.best_params

## Entraînement final du modèle

Pour l'instant, meilleurs résultats pour 'nb_hidden_layers': 3, 'nb_hidden_channels': 32, 'z_dim': 7

In [ ]:
saving_path = Path.cwd() / "checkpoints" / "local" / "Version_peut_etre_finale"

max_epochs = 70
batch_size = 4
competition_table_period = timedelta(days=365)
train_test_lim = datetime(2023, 1, 1)
hidden_part = 0.15
num_workers = 4

net_hyperparams = dict(
    in_channels_enc=6,
    hidden_channels_enc=32,
    nb_hidden_layers_enc=3,
    zdim_line=4,
    zdim_col=4,
    hidden_channels_dec=32,
    nb_hidden_layers_dec=3,
    huber_loss_delta=0.0379313270566344,
    warmup_epochs=3,
    total_epochs=max_epochs,
)

checkpoint_callback = ModelCheckpoint(
    monitor="Median absolute prediction error (s)",
    mode="min",
    save_top_k=1,
    dirpath=saving_path.parent,
    filename=saving_path.name,
)
tqdm_bar = TQDMProgressBar(refresh_rate=1)
logger = TensorBoardLogger(
    save_dir=Path.cwd() / "lightning_logs", name=saving_path.name
)
trainer = Trainer(
    callbacks=[checkpoint_callback, tqdm_bar],
    max_epochs=max_epochs,
    logger=logger,
)

data_module = CompetitionDataModule(
    files_saving_path,
    batch_size=batch_size,
    competition_table_period=competition_table_period,
    train_test_lim=train_test_lim,
    hidden_part=hidden_part,
    num_workers=num_workers,
)

model = CompetitionAutoencoderNet(**net_hyperparams)

trainer.fit(model, data_module)


# model.load_data(
#     starting_date_train=starting_date_train,
#     ending_date_train=ending_date_train,
#     starting_date_val=starting_date_val,
#     ending_date_val=ending_date_val,
# )

# Brouillon

In [ ]:
dir(model)